<a href="https://colab.research.google.com/github/Dominic0311/LA-Live-Events-Analytics/blob/main/Revenue_Prediction_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import sqlite3
import pandas as pd

# 1. Read your CSV with latin1 encoding to handle special characters
raw_df = pd.read_csv("Taylor_Train.csv", encoding="latin1")

# 2. Standardize column names
raw_df.columns = [
    "City",
    "Country",
    "Venue",
    "Opening_Acts",
    "Attendance_Raw",
    "Revenue_Raw",
    "Tour",
]

# 3. Create an in-memory SQL database and load the table
conn = sqlite3.connect(":memory:")
raw_df.to_sql("concerts_raw", conn, index=False, if_exists="replace")

# 4. Run SQL to clean non-numeric characters ($ , / N/A) and extract metrics
clean_sql = """
SELECT
    City,
    Country,
    Venue,
    Tour,
    -- Remove '$' and ',' from Revenue and convert to number
    CAST(REPLACE(REPLACE(Revenue_Raw, '$', ''), ',', '') AS INTEGER) AS Revenue,

    -- Split Attendance string before '/' for Tickets Sold
    CAST(REPLACE(SUBSTR(Attendance_Raw, 1, INSTR(Attendance_Raw, '/') - 1), ',', '') AS INTEGER) AS Tickets_Sold,

    -- Split Attendance string after '/' for Capacity/Available Tickets
    CAST(REPLACE(SUBSTR(Attendance_Raw, INSTR(Attendance_Raw, '/') + 1), ',', '') AS INTEGER) AS Tickets_Available
FROM concerts_raw
WHERE Revenue_Raw IS NOT NULL
  AND Revenue_Raw NOT LIKE '%N/A%'
  AND Revenue_Raw NOT LIKE '%\ufffd%'
  AND Attendance_Raw LIKE '%/%';
"""

# Execute the SQL Query
cleaned_df = pd.read_sql_query(clean_sql, conn)

# Display the first 5 cleaned SQL results directly in Colab
print("--- CLEANED SQL OUTPUT SAMPLE ---")
print(cleaned_df.head())

# 5. Export cleaned dataset to CSV for Power BI
cleaned_df.to_csv("taylor_swift_concerts_cleaned.csv", index=False)
print(
    "\nSuccess! 'taylor_swift_concerts_cleaned.csv' is ready in your Colab files panel."
)

--- CLEANED SQL OUTPUT SAMPLE ---
               City        Country                                 Venue  \
0        Evansville  United States             Roberts Municipal Stadium   
1         Jonesboro  United States                    Convocation Center   
2         St. Louis  United States                      Scottrade Center   
3  North Charleston  United States             North Charleston Coliseum   
4      Jacksonville  United States  Jacksonville Veterans Memorial Arena   

            Tour  Revenue  Tickets_Sold  Tickets_Available  
0  Fearless_Tour   360617          7463               7463  
1  Fearless_Tour   340328          7822               7822  
2  Fearless_Tour   650420         13764              13764  
3  Fearless_Tour   398154          8751               8751  
4  Fearless_Tour   507012         11072              11072  

Success! 'taylor_swift_concerts_cleaned.csv' is ready in your Colab files panel.


In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import pandas as pd

# 1. Load the cleaned data from your earlier step
df = pd.read_csv("taylor_swift_concerts_cleaned.csv")

# 2. Define Features (X) and Target Variable (y)
X = df[["Tickets_Sold", "Tickets_Available"]]
y = df["Revenue"]

# 3. Split into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Train a Machine Learning Model
model = LinearRegression()
model.fit(X_train, y_train)

# 5. Evaluate Model Accuracy (R-squared score)
score = model.score(X_test, y_test)
print(
    f"Model Training Complete! Accuracy Score (R^2): {score * 100:.2f}%"
)

# 6. Predict Revenue for a new 50,000 capacity stadium concert
sample_venue = pd.DataFrame(
    [[50000, 50000]], columns=["Tickets_Sold", "Tickets_Available"]
)
predicted_revenue = model.predict(sample_venue)[0]
print(
    f"Predicted Revenue for 50,000 Capacity Show: ${predicted_revenue:,.2f}"
)

Model Training Complete! Accuracy Score (R^2): 92.25%
Predicted Revenue for 50,000 Capacity Show: $5,189,552.92
